# 18d · DMID overlay sheets (robust) + CBIS-DDSM descriptors
**Runtime: CPU, High-RAM** (about 30–50 min). Uses `results/p2_dmid_audit18b/assignments.csv` written by 18c (cells 2–3 already done there).
Draws the overlay sheets (no model predictions) with a crop window that always stays inside the image and covers both the listed circle (1.6 r) and the assigned region; records whose window would be empty get a blank panel and are listed. Then writes `audit_template.csv` and the CBIS-DDSM descriptors.

In [ ]:
!pip -q install imagecodecs
from google.colab import drive
drive.mount('/content/drive')
import os, glob, subprocess, requests, json, re, gc
import numpy as np, pandas as pd, cv2, tifffile
from PIL import Image
P = '/content/drive/MyDrive/quantum-placement'
OUT = f'{P}/results/p2_dmid_audit18b'; os.makedirs(f'{OUT}/sheets', exist_ok=True)
DEDUP = f'{P}/results/p2_analysis_v2/dmid_dedup.csv'
JITMETA = f'{P}/data/external/dmid_mass_256_jit/roi/metadata.csv'
RAW, LOCAL = '/content/dmid_raw', '/content/dmid'; os.makedirs(RAW, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
print(os.path.exists(DEDUP), os.path.exists(JITMETA))

In [ ]:
# 1 · DMID download (figshare 24522883) — same files and parsing as notebook 11
art = requests.get('https://api.figshare.com/v2/articles/24522883', timeout=60).json()
for f in art['files']:
    if not any(k in f['name'] for k in ('TIFF', 'ROI', 'Metadata')): continue
    dst = f"{RAW}/{f['name']}"
    if not os.path.exists(dst) or os.path.getsize(dst) != f['size']:
        with requests.get(f['download_url'], stream=True, timeout=900) as r:
            r.raise_for_status()
            with open(dst, 'wb') as o:
                for ch in r.iter_content(1 << 20): o.write(ch)
    if dst.endswith('.zip') and not os.path.isdir(f"{LOCAL}/{f['name'][:-4]}"):
        subprocess.run(['unzip', '-q', '-o', dst, '-d', LOCAL], check=False)
def read_tif(p):
    try: a = tifffile.imread(p)
    except Exception: a = np.array(Image.open(p))
    a = np.asarray(a)
    if a.ndim == 3 and a.shape[0] in (1, 3, 4) and a.shape[-1] not in (1, 3, 4): a = np.moveaxis(a, 0, -1)
    if a.ndim == 3: a = a[..., :3].mean(-1) if a.shape[-1] >= 3 else a[..., 0]
    return a
def to_uint8(img):
    img = img.astype(np.float32); nz = img[img > 0]
    lo, hi = (np.percentile(nz, 1), np.percentile(nz, 99.5)) if nz.size else (img.min(), img.max())
    return (np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)
info_path = glob.glob(f'{LOCAL}/**/Info.txt', recursive=True)[0]
rows = []
for line in open(info_path, errors='ignore'):
    p = line.split()
    if len(p) >= 3 and p[0].upper().startswith('IMG'):
        def _f(s):
            try: return float(s)
            except ValueError: return np.nan
        rows.append(dict(img=p[0].upper(), tissue=p[1], cls=p[2], sev=p[3] if len(p) > 3 else None,
                         x=_f(p[4]) if len(p) > 6 else np.nan, y=_f(p[5]) if len(p) > 6 else np.nan, r_raw=_f(p[6]) if len(p) > 6 else np.nan))
info = pd.DataFrame(rows)
IMG_DIR = glob.glob(f'{LOCAL}/**/TIFF Images', recursive=True)[0]
ROI_DIR = glob.glob(f'{LOCAL}/**/ROI Masks', recursive=True)[0]
mass = info[info.cls.isin(['CIRC', 'SPIC', 'MISC']) & info.x.notna() & info.y.notna()].copy()
mass['r'] = mass.r_raw.fillna(50).clip(lower=20)
mass = mass.reset_index(drop=True)            # index i -> roi_id IMGxxx_i, exactly as in notebook 11
mass['roi_id'] = [f'{a}_{i}' for i, a in zip(mass.index, mass.img)]
print('listed masses with coordinates:', len(mass), '| images:', mass.img.nunique())

In [ ]:
# 2 · Load the audit table from 18c and define the label helper
A = pd.read_csv(f'{OUT}/assignments.csv', keep_default_na=True)
A['comp_ids'] = A['comp_ids'].fillna('').astype(str); A['flags'] = A['flags'].fillna('').astype(str)
def labels_of(img_id, with_image=False):
    m = read_tif(f'{ROI_DIR}/{img_id}.tif') > 0; im = read_tif(f'{IMG_DIR}/{img_id}.tif')
    if im.shape != m.shape: m = cv2.resize(m.astype(np.uint8), (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    n, lab, st, cen = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    out = (n, lab, st, cen, to_uint8(im) if with_image else None); del m, im; return out
HAS_MASK = {i for i in A.img.unique() if os.path.exists(f'{ROI_DIR}/{i}.tif')}
print(A.status.value_counts().to_string())

In [ ]:
# 3 · Overlay sheets (blind to model output): 24 panels per sheet, robust crop window
import matplotlib.pyplot as plt
EMPTY = []
def panel(ax, r):
    if r.img in HAS_MASK: n, lab, st, _, im8 = labels_of(r.img, with_image=True)
    else:
        im8 = to_uint8(read_tif(f'{IMG_DIR}/{r.img}.tif')); lab = np.zeros_like(im8, dtype=np.int32)
    H, W = im8.shape
    xs = [r.x - 1.6 * r.r, r.x + 1.6 * r.r]; ys = [r.y - 1.6 * r.r, r.y + 1.6 * r.r]
    if r.n_comp > 0 and pd.notna(r.x0): xs += [r.x0, r.x1]; ys += [r.y0, r.y1]
    cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
    half = max(160, 0.55 * max(max(xs) - min(xs), max(ys) - min(ys)))
    x0, x1 = int(np.clip(cx - half, 0, W)), int(np.clip(cx + half, 0, W)); y0, y1 = int(np.clip(cy - half, 0, H)), int(np.clip(cy + half, 0, H))
    inside = 0 <= r.x < W and 0 <= r.y < H
    ttl = f"{r.roi_id} {r.cls}/{r.sev} ids[{r.comp_ids or '-'}]\n{r.status}" + (f" | {r['flags']}" if r['flags'] else '') + ('' if inside else ' | LISTED CENTRE OUTSIDE IMAGE')
    if x1 - x0 < 2 or y1 - y0 < 2:
        EMPTY.append(r.roi_id); ax.axis('off'); ax.set_title(ttl + '\n(EMPTY WINDOW)', fontsize=5.2); return
    rgb = cv2.cvtColor(np.ascontiguousarray(im8[y0:y1, x0:x1]), cv2.COLOR_GRAY2RGB); s = 360 / max(rgb.shape[:2]); rgb = cv2.resize(rgb, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    sub = cv2.resize(lab[y0:y1, x0:x1].astype(np.float32), (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST).astype(np.int32)
    assigned = set(map(int, r.comp_ids.split())) if r.comp_ids else set()
    for k in np.unique(sub):
        if k == 0: continue
        cs, _ = cv2.findContours((sub == k).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(rgb, cs, -1, (255, 40, 40) if k in assigned else (0, 220, 255), 2 if k in assigned else 1)
        yy, xx = np.nonzero(sub == k); cv2.putText(rgb, str(k), (int(xx.mean()), int(yy.mean())), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 40, 40) if k in assigned else (0, 220, 255), 1)
    c = (int((r.x - x0) * s), int((r.y - y0) * s))
    cv2.circle(rgb, c, max(1, int(r.r * s)), (255, 230, 0), 1); cv2.drawMarker(rgb, c, (255, 230, 0), cv2.MARKER_CROSS, 10, 1)
    for t in np.linspace(0, 2 * np.pi, 48, endpoint=False)[::2]:
        a, b = t, t + np.pi / 48
        p1 = (int(c[0] + 1.5 * r.r * s * np.cos(a)), int(c[1] + 1.5 * r.r * s * np.sin(a))); p2 = (int(c[0] + 1.5 * r.r * s * np.cos(b)), int(c[1] + 1.5 * r.r * s * np.sin(b)))
        cv2.line(rgb, p1, p2, (255, 230, 0), 1)
    del lab, im8, sub; ax.imshow(rgb); ax.axis('off'); ax.set_title(ttl, fontsize=5.2)
order = pd.concat([A[A.status == 'retained'], A[A.status != 'retained']])
for j in range(0, len(order), 24):
    part = order.iloc[j:j + 24]; fig, axs = plt.subplots(4, 6, figsize=(18, 13))
    for a in axs.ravel(): a.axis('off')
    for a, (_, r) in zip(axs.ravel(), part.iterrows()):
        try: panel(a, r)
        except Exception as e: EMPTY.append(r.roi_id); a.set_title(f'{r.roi_id}: drawing failed ({type(e).__name__})', fontsize=5.2)
    plt.tight_layout(); fig.savefig(f'{OUT}/sheets/sheet_{j // 24 + 1:02d}.png', dpi=80); plt.close(fig); gc.collect()
T = order[['roi_id', 'img', 'cls', 'sev', 'status', 'comp_ids', 'flags', 'x', 'y', 'r']].copy()
T['sheet'] = [f'sheet_{k // 24 + 1:02d}' for k in range(len(T))]
T['decision (accept/correct/exclude/uncertain)'] = ''; T['corrected comp_ids'] = ''; T['reason'] = ''; T['reviewer'] = ''
T.to_csv(f'{OUT}/audit_template.csv', index=False)
print('sheets:', len([f for f in os.listdir(f'{OUT}/sheets') if f.endswith('.png')]), '| records:', len(T), '| empty/failed panels:', EMPTY)

In [ ]:
# 5 · CBIS-DDSM descriptors of the 400 selected ROIs vs all eligible mass ROIs (public TCIA CSVs)
URLS = {'train': ['https://www.cancerimagingarchive.net/wp-content/uploads/mass_case_description_train_set.csv',
                  'https://wiki.cancerimagingarchive.net/download/attachments/22516629/mass_case_description_train_set.csv'],
        'test': ['https://www.cancerimagingarchive.net/wp-content/uploads/mass_case_description_test_set.csv',
                 'https://wiki.cancerimagingarchive.net/download/attachments/22516629/mass_case_description_test_set.csv']}
parts = []
for split, us in URLS.items():
    for u in us:
        try:
            r = requests.get(u, timeout=60); r.raise_for_status(); df = pd.read_csv(io.StringIO(r.text)) if False else pd.read_csv(__import__('io').StringIO(r.text))
            df['split'] = 'Training' if split == 'train' else 'Test'; parts.append(df); print('ok', u); break
        except Exception as e: print('failed', u, e)
if len(parts) == 2:
    C = pd.concat(parts, ignore_index=True)
    C['roi_id'] = 'Mass-' + C.split + '_' + C.patient_id + '_' + C['left or right breast'] + '_' + C['image view'] + '_' + C['abnormality id'].astype(str)
    sel = pd.read_csv(f'{P}/data/processed/cbis_mass_256_jit/folds.csv').roi_id
    C['selected'] = C.roi_id.isin(set(sel))
    print('matched selected ROIs:', int(C.selected.sum()), 'of', len(sel), '| CSV mass ROIs:', len(C))
    C[['roi_id', 'patient_id', 'split', 'pathology', 'assessment', 'subtlety', 'mass shape', 'mass margins', 'selected']].to_csv(f'{OUT}/cbis_descriptors.csv', index=False)
    for col in ['pathology', 'assessment']:
        print(pd.crosstab(C[col], C.selected, normalize='columns').round(3).rename(columns={False: 'not selected', True: 'selected'}).to_string())
    print(C.groupby('selected').subtlety.describe().round(2).to_string())
else:
    print('CBIS-DDSM description CSVs not reachable; upload mass_case_description_train_set.csv and _test_set.csv to', OUT)